# E6 · The test split's ground truth: four-class masks for the benchmark's optical test pairs

Task **E6** (built 4 Nov 2026). **CPU only:** set the accelerator to None, so it uses no GPU hours.

**Why**
- Every segmentation score so far is on our 645 validation pairs (605 clean), which come from DisasterM3's training data.
- DisasterM3's test images (its question benchmark) carry building masks too. Its "Referring Expression Segmentation" task gives, for each optical test image, one black-and-white mask per damage class it asks about: intact, damaged or destroyed.
- Merged into one four-class map per image, they would make a second, larger and unseen test set: 2,067 optical before-and-after pairs, 414 of them from Turkey's 2023 earthquake (validation has 64).
- Before any model is scored on them, three things must hold: the masks are like the training masks (values, size, no overlaps), no test pair copies an image our segmentation models trained on, and the outlines are drawn alike.

**What it does**
1. Rebuilds the 6,443 segmentation pairs and the 645-pair validation split as every D4 run did, and checks it: 645 pairs, 560 from xBD, 1,808,111 destroyed pixels.
2. Lists the benchmark's optical building-damage segmentation entries, one test pair per after-image, and finds each pair's after-image, before-image and masks.
3. Checks every mask (values, size against its after-image, masks of classes with no question, overlaps), merges each pair's masks with the training rule, and saves one four-class mask per test pair: 0 background, 1 intact, 2 damaged, 3 destroyed.
4. Flags test pairs that copy a segmentation-pool image, from E5 step 1's fingerprints (`bench_copies.csv`).
5. Compares building outlines between test and validation within the same disaster events.

**Decision rules and expectations, fixed before the run (4 Nov)**
1. **Inventory:** 2,067 optical test pairs (1,030 with one mask, 558 with two, 479 with three; counted from `benchmark_release.json` on Abrar's PC), every after-image, before-image and asked-about mask found. A pair with a file missing is listed and left out.
2. **Masks like the training masks:** only the values 0 and 255; 1024 × 1024, the size of the after-image, with the before-image the same size; masks of classes an image has no question about are empty. The mask checks all held on 600 training images checked on Abrar's PC. A pair failing one is listed and left out.
3. **Overlaps:** expected 0 pixels (0 of 48.9 million building pixels on those 600 training images). Masks are merged with the training rule (R11's Cell 4: intact, then damaged, then destroyed, a later class overwriting an earlier one). If more than 0.5% of building pixels overlap, the README says the merged masks aren't like the training ones.
4. **Copies:** a test pair is left out if its after- or before-image is a same-pixel copy or a near copy (E5 step 1's tiers) of an image in the 6,443 segmentation pairs, training or validation. Copies of Instruct images outside those pairs are kept, since no segmentation model trained on them. Computed on Abrar's PC on 4 Nov from the same files: **57 left out (56 copy a training pair, 1 a validation pair), 2,010 clean.** This run must reproduce it.
5. **Drawn alike:** for each event with at least 20 pairs in both validation and test, the ratio test / validation of the median building-piece area (a piece: 8-connected pixels of one class, at least 20 px, as in R16c step 5b). The outlines count as drawn alike if the ratio lies within 0.67–1.5 for at least 80% of those events. No number was predicted.

**How to run on Kaggle** (account 2; any account that can read both datasets works)
1. Session options: Accelerator **None**. Internet on, unless input 3 is attached.
2. Inputs:
   1. `disasterm3-mirror` (abrarmohammedtanzim);
   2. `disasterm3-bench-mirror-v2` (redwannewaz);
   3. optional: the output of `e5-step1-hash-check` (account 2's notebook). Without it, Cell 5 downloads the same `bench_copies.csv` from the team's GitHub repository.
3. Run All with `SMOKE_TEST = True` (2–3 minutes: every check on the split and the inventory, then 40 test and 40 validation pairs). Then set `SMOKE_TEST = False` and Save & Run All (Commit), about 10–15 minutes.
4. Outputs in `/kaggle/working/E6/`:
   - `README.md` and `summary.json`;
   - `test_pairs.csv`: one row per optical test pair, every check, and whether it's clean;
   - `test_clean_pairs.csv`: the clean pairs only (after-image, before-image, mask file);
   - `style_by_event.csv` and `val_pairs_stats.csv`;
   - `test_masks_4class.zip`: the merged masks, one PNG per test pair, named after its after-image.

In [1]:
# ── Cell 1: Configuration ─────────────────────────────────────────────────
import csv
import glob
import json
import multiprocessing as mp
import os
import re
import shutil
import time
import urllib.request
from collections import Counter, defaultdict
from concurrent.futures import ProcessPoolExecutor
from pathlib import Path

import cv2
import numpy as np
from PIL import Image

SMOKE_TEST = True                      # True = all split and inventory checks, then 40 test and 40 validation pairs
INPUT_ROOT = Path("/kaggle/input")
DATA_ROOT = INPUT_ROOT / "datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct"
MANIFEST_PATH = DATA_ROOT / "train_release.json"
OUT_DIR = Path("/kaggle/working/E6")
MASK_OUT = OUT_DIR / "test_masks_4class"
GITHUB_BENCH_COPIES = ("https://raw.githubusercontent.com/SmashNdashH/CSE499A.15-Msrb-/main/"
                       "Essential_task_delegation/Champion_Task/E5-results/step1/bench_copies.csv")
WORKERS = os.cpu_count() or 4
MIN_PIECE_PX = 20                      # a building piece: 8-connected pixels of one class, at least 20 px (R16c step 5b)
CLASS_FOLDERS = ["building_intact_mask", "building_damaged_mask", "building_destroyed_mask"]   # classes 1, 2, 3

EXPECTED = {"pairs": 6443, "val": 645, "val_xbd": 560, "val_destroyed_px": 1808111,
            "test_pairs": 2067, "test_by_masks": {1: 1030, 2: 558, 3: 479},
            "left_out_copies": 57, "left_out_training": 56, "left_out_validation": 1, "clean": 2010}
OVERLAP_LIMIT = 0.005                  # rule 3: overlapping pixels as a share of building pixels
STYLE_MIN_PAIRS, STYLE_LOW, STYLE_HIGH, STYLE_SHARE = 20, 0.67, 1.5, 0.80   # rule 5

assert MANIFEST_PATH.exists(), f"No manifest at {MANIFEST_PATH}: attach abrarmohammedtanzim/disasterm3-mirror"
_bench = [Path(p).parent for p in sorted(glob.glob(str(INPUT_ROOT / "**" / "benchmark_release.json"), recursive=True))
          if (Path(p).parent / "test_images").is_dir()]
if not _bench:
    raise FileNotFoundError("Attach redwannewaz/disasterm3-bench-mirror-v2 (benchmark_release.json next to test_images/)")
BENCH_ROOT = _bench[0]
if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)             # a rerun starts clean, so no mask from an earlier run is zipped
MASK_OUT.mkdir(parents=True)
print(f"✓ E6{'  [SMOKE TEST]' if SMOKE_TEST else ''}: Instruct {DATA_ROOT}; benchmark {BENCH_ROOT}; "
      f"{WORKERS} worker(s); outputs {OUT_DIR}")

✓ E6  [SMOKE TEST]: Instruct /kaggle/input/datasets/abrarmohammedtanzim/disasterm3-mirror/DisasterM3_Instruct; benchmark /kaggle/input/datasets/redwannewaz/disasterm3-bench-mirror-v2; 4 worker(s); outputs /kaggle/working/E6


In [2]:
# ── Cell 2: The 645 validation pairs, rebuilt as every D4 run did (R11's Cell 4 and split), and checked ──
from sklearn.model_selection import train_test_split


def resolve_path(rel_path, root=DATA_ROOT, image_dir="train_images"):
    """R11's path resolution; root=BENCH_ROOT, image_dir='test_images' resolves benchmark paths the same way."""
    if not rel_path:
        return None
    rel_path = rel_path.replace("\\", "/")
    filename = Path(rel_path).name
    for c in (root / rel_path, root / image_dir / rel_path, root / image_dir / image_dir / filename,
              root / "masks" / rel_path, root / "masks" / "masks" / filename):
        if c.exists():
            return str(c)
    return None


def name_of(path):
    return Path(str(path).replace("\\", "/")).name


def folder_class(folder):
    """'train_building_damaged_mask' or 'test_building_damaged_mask' -> 2."""
    for k, suffix in enumerate(CLASS_FOLDERS, 1):
        if folder.endswith(suffix):
            return k
    return None


def bda_optical(entries):
    """Optical building-damage segmentation entries grouped by after-image, in manifest order (R11's filter)."""
    by_image = {}
    for e in entries:
        if (e.get("task") != "Referring Expression Segmentation"
                or e.get("cls_description") != "Building Damage Assessment" or e.get("image_type") != "Optical"):
            continue
        gt = e.get("ground_truth", "")
        d = by_image.setdefault(e["post_image_path"], {"pre": e.get("pre_image_path"), "masks": {}})
        d["masks"][Path(gt.replace("\\", "/")).parent.name] = gt
    return by_image


XBD_NAME = re.compile(r"^(.+?)_(\d{8})_post_disaster")
OTHER_NAME = re.compile(r"^(.+?)_post")


def event_and_source(name):
    """E1's rule: 'hurricane_florence_00000261_post_disaster.png' -> ('hurricane_florence', 'xBD')."""
    m = XBD_NAME.match(name)
    if m:
        return m.group(1), "xBD"
    m = OTHER_NAME.match(name)
    return re.sub(r"\d+$", "", m.group(1) if m else Path(name).stem), "other"


with open(MANIFEST_PATH, encoding="utf-8") as f:
    train_manifest = json.load(f)
pool = []                              # R11's `pairs`: an after-image that resolves, with at least one mask that does
for post, d in bda_optical(train_manifest).items():
    img = resolve_path(post)
    masks = {folder_class(fo): resolve_path(rel) for fo, rel in d["masks"].items()}
    masks = {k: p for k, p in masks.items() if p}
    if img and masks:
        pool.append({"post": name_of(post), "pre": name_of(d["pre"]), "image_path": img,
                     "referenced": sorted(masks), "mask_name": name_of(next(iter(masks.values()))), "masks": masks})
train_pool, val_pool = train_test_split(pool, test_size=0.1, random_state=42)


def destroyed_px(row):
    m = cv2.imread(row["masks"][3], cv2.IMREAD_GRAYSCALE) if 3 in row["masks"] else None
    return int((m > 0).sum()) if m is not None else 0   # class 3 is merged last, so it keeps every one of its pixels


val_destroyed = sum(destroyed_px(r) for r in val_pool)
val_xbd = sum(event_and_source(r["post"])[1] == "xBD" for r in val_pool)
print(f"✓ Pairs {len(pool):,}: training {len(train_pool):,}, validation {len(val_pool)} ({val_xbd} xBD); "
      f"destroyed pixels in validation {val_destroyed:,}")
assert (len(pool), len(val_pool), val_xbd) == (EXPECTED["pairs"], EXPECTED["val"], EXPECTED["val_xbd"]), "not the D4 split"
assert val_destroyed == EXPECTED["val_destroyed_px"], "the validation masks differ from the ones every D4 run scored"
TRAIN_FILES = {r[k] for r in train_pool for k in ("post", "pre")}
VAL_FILES = {r[k] for r in val_pool for k in ("post", "pre")}
print("✓ The split matches every D4 run")

✓ Pairs 6,443: training 5,798, validation 645 (560 xBD); destroyed pixels in validation 1,808,111
✓ The split matches every D4 run


In [3]:
# ── Cell 3: The benchmark's optical building-damage segmentation entries: one test pair per after-image ──
with open(BENCH_ROOT / "benchmark_release.json", encoding="utf-8") as f:
    bench = json.load(f)
sar_images = len({e["post_image_path"] for e in bench if e.get("task") == "Referring Expression Segmentation"
                  and e.get("cls_description") == "Building Damage Assessment" and e.get("image_type") == "SAR"})


def all_masks(asked, mask_name, prefix, root, image_dir):
    """Every class's mask file for one image: the asked-about ones as resolved, the others looked up by name."""
    out = {}
    for k, suffix in enumerate(CLASS_FOLDERS, 1):
        out[k] = asked.get(k) or resolve_path(f"masks/{prefix}{suffix}/{mask_name}", root, image_dir)
    return out


test = []
for post, d in bda_optical(bench).items():
    asked = {folder_class(fo): resolve_path(rel, BENCH_ROOT, "test_images") for fo, rel in d["masks"].items()}
    names = {name_of(rel) for rel in d["masks"].values()}
    row = {"post": name_of(post), "pre": name_of(d["pre"]), "mask_name": sorted(names)[0],
           "image_path": resolve_path(post, BENCH_ROOT, "test_images"),
           "pre_path": resolve_path(d["pre"], BENCH_ROOT, "test_images"),
           "referenced": sorted(asked), "problems": []}
    row["masks"] = all_masks({k: p for k, p in asked.items() if p}, row["mask_name"], "test_", BENCH_ROOT, "test_images")
    if not row["image_path"]:
        row["problems"].append("after-image missing")
    if not row["pre_path"]:
        row["problems"].append("before-image missing")
    row["problems"] += [f"asked-about mask missing (class {k})" for k, p in asked.items() if not p]
    if len(names) > 1:
        row["problems"].append("masks named differently")
    test.append(row)

for r in val_pool:                     # the same lookups for validation, so both get the same checks
    r["masks"] = all_masks(r["masks"], r["mask_name"], "train_", DATA_ROOT, "train_images")
    r["pre_path"] = resolve_path(r["pre"])
    r["problems"] = [] if r["pre_path"] else ["before-image missing"]
assert not any(r["problems"] for r in val_pool), "a validation pair has no before-image (every D4 run had all 645)"

by_masks = Counter(len(r["referenced"]) for r in test)
inventory_problems = [r for r in test if r["problems"]]
print(f"✓ Optical test pairs: {len(test):,} (expected {EXPECTED['test_pairs']:,}); by number of masks asked about: "
      f"{dict(sorted(by_masks.items()))} (expected {EXPECTED['test_by_masks']}); SAR after-images skipped: {sar_images:,}")
print(f"  Pairs with a missing or mismatched file: {len(inventory_problems)}")
for r in inventory_problems[:10]:
    print(f"    {r['post']}: {'; '.join(r['problems'])}")
print(f"  Before-images shared by two or more test pairs: "
      f"{sum(c > 1 for c in Counter(r['pre'] for r in test).values())}")

✓ Optical test pairs: 2,067 (expected 2,067); by number of masks asked about: {1: 1030, 2: 558, 3: 479} (expected {1: 1030, 2: 558, 3: 479}); SAR after-images skipped: 829
  Pairs with a missing or mismatched file: 0
  Before-images shared by two or more test pairs: 0


In [4]:
# ── Cell 4: Every mask checked, merged with the training rule, and its building pieces measured ──
def image_size(path):
    """(width, height) from the file header, closing the file at once; None if there is no file."""
    if not path:
        return None
    with Image.open(path) as im:
        return im.size


def check_and_merge(job):
    """One pair -> its mask checks, overlaps, class pixels and building pieces; a test pair's merged mask is saved."""
    cv2.setNumThreads(1)
    kind, r = job
    out = {"post": r["post"], "values_ok": True, "size_ok": True, "pre_size_ok": False, "unreadable": [],
           "unasked_nonempty": [], "overlap_px": 0, "building_px": 0, "class_px": [0, 0, 0], "pieces": [0, 0, 0],
           "piece_areas": [], "saved": False}
    size = image_size(r.get("image_path"))
    out["pre_size_ok"] = size is not None and image_size(r.get("pre_path")) == size
    asked = {}
    for k, p in r["masks"].items():
        if not p:
            continue
        m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if m is None:
            out["unreadable"].append(k)
            continue
        if not set(np.unique(m).tolist()) <= {0, 255}:
            out["values_ok"] = False
        if m.shape != (1024, 1024) or size is None or (m.shape[1], m.shape[0]) != size:
            out["size_ok"] = False
        if k in r["referenced"]:
            asked[k] = m > 0
        elif m.any():
            out["unasked_nonempty"].append(k)
    if not asked:
        return out
    shape = asked[min(asked)].shape
    count = np.zeros(shape, np.uint8)
    merged = np.zeros(shape, np.uint8)
    for k in (1, 2, 3):                    # R11's rule: a later class overwrites an earlier one
        if k in asked and asked[k].shape == shape:
            count += asked[k]
            merged[asked[k]] = k
    out["overlap_px"] = int((count > 1).sum())
    out["building_px"] = int((count > 0).sum())
    for k in (1, 2, 3):
        cls = (merged == k).astype(np.uint8)
        out["class_px"][k - 1] = int(cls.sum())
        areas = cv2.connectedComponentsWithStats(cls, connectivity=8)[2][1:, cv2.CC_STAT_AREA]
        areas = areas[areas >= MIN_PIECE_PX]
        out["pieces"][k - 1] = int(len(areas))
        out["piece_areas"] += areas.tolist()
    if kind == "test":
        out["saved"] = bool(cv2.imwrite(str(MASK_OUT / r["post"]), merged))
    return out


def passes(s):
    """Rule 2 for one pair: masks of 0 and 255 only, 1024 x 1024 like the after-image, before-image the same size,
    every mask readable, masks of classes with no question empty."""
    return s["values_ok"] and s["size_ok"] and s["pre_size_ok"] and not s["unreadable"] and not s["unasked_nonempty"]


def run_all(jobs, label):
    # "fork" lets the workers call check_and_merge(), which lives in this notebook (as in E5 step 1);
    # with one CPU, or where fork doesn't exist, the pairs run one after another in this process
    ctx = mp.get_context("fork") if "fork" in mp.get_all_start_methods() else None
    ex = ProcessPoolExecutor(max_workers=WORKERS, mp_context=ctx) if WORKERS > 1 and ctx else None
    out, t0 = [], time.time()
    for i, res in enumerate(ex.map(check_and_merge, jobs, chunksize=8) if ex else map(check_and_merge, jobs), 1):
        out.append(res)
        if i % 500 == 0 or i == len(jobs):
            print(f"  {label}: {i:,}/{len(jobs):,} ({(time.time() - t0) / 60:.1f} min)")
    if ex:
        ex.shutdown()
    return {s["post"]: s for s in out}


def mask_summary(stats):
    s = list(stats.values())
    building = sum(x["building_px"] for x in s)
    areas = [a for x in s for a in x["piece_areas"]]
    return {"pairs": len(s), "fail_rule_2": sum(not passes(x) for x in s),
            "values_not_0_or_255": sum(not x["values_ok"] for x in s),
            "mask_not_1024_or_not_image_size": sum(not x["size_ok"] for x in s),
            "before_image_other_size": sum(not x["pre_size_ok"] for x in s),
            "unreadable_mask": sum(bool(x["unreadable"]) for x in s),
            "unasked_mask_not_empty": sum(bool(x["unasked_nonempty"]) for x in s),
            "pairs_with_overlap": sum(x["overlap_px"] > 0 for x in s),
            "overlap_px": sum(x["overlap_px"] for x in s), "building_px": building,
            "overlap_share": round(sum(x["overlap_px"] for x in s) / max(building, 1), 6),
            "class_px": [sum(x["class_px"][k] for x in s) for k in range(3)],
            "pieces": [sum(x["pieces"][k] for x in s) for k in range(3)],
            "median_piece_area_px": float(np.median(areas)) if areas else None}


test_jobs = [r for r in test if not r["problems"]]
val_jobs = val_pool
if SMOKE_TEST:
    test_jobs, val_jobs = test_jobs[:40], val_pool[:40]
TEST_STATS = run_all([("test", r) for r in test_jobs], "test")
VAL_STATS = run_all([("val", r) for r in val_jobs], "validation")
TEST_MASKS, VAL_MASKS = mask_summary(TEST_STATS), mask_summary(VAL_STATS)
for label, m in (("Test", TEST_MASKS), ("Validation", VAL_MASKS)):
    print(f"✓ {label}: {m['pairs']:,} pairs checked; failing rule 2: {m['fail_rule_2']} "
          f"(values {m['values_not_0_or_255']}, size {m['mask_not_1024_or_not_image_size']}, "
          f"before-image size {m['before_image_other_size']}, unreadable {m['unreadable_mask']}, "
          f"unasked mask not empty {m['unasked_mask_not_empty']})")
    print(f"  overlapping pixels {m['overlap_px']:,} of {m['building_px']:,} building pixels ({m['overlap_share']:.4%}) "
          f"in {m['pairs_with_overlap']} pairs; pieces intact / damaged / destroyed {m['pieces']}; "
          f"median piece {m['median_piece_area_px']} px")

  test: 40/40 (0.1 min)
  validation: 40/40 (0.1 min)
✓ Test: 40 pairs checked; failing rule 2: 0 (values 0, size 0, before-image size 0, unreadable 0, unasked mask not empty 0)
  overlapping pixels 0 of 12,953,594 building pixels (0.0000%) in 0 pairs; pieces intact / damaged / destroyed [3301, 337, 68]; median piece 1569.0 px
✓ Validation: 40 pairs checked; failing rule 2: 0 (values 0, size 0, before-image size 0, unreadable 0, unasked mask not empty 0)
  overlapping pixels 0 of 4,062,969 building pixels (0.0000%) in 0 pairs; pieces intact / damaged / destroyed [1193, 174, 88]; median piece 1273.0 px


In [5]:
# ── Cell 5: Test pairs that copy a segmentation-pool image (E5 step 1's fingerprints, rule 4) ──
found = sorted(glob.glob(str(INPUT_ROOT / "**" / "bench_copies.csv"), recursive=True))
if found:
    copies_path, copies_source = Path(found[0]), f"E5 step 1's Kaggle output ({found[0]})"
else:
    copies_path, copies_source = OUT_DIR / "bench_copies.csv", "the team's GitHub repository"
    try:
        urllib.request.urlretrieve(GITHUB_BENCH_COPIES, copies_path)
    except OSError as ex:
        raise RuntimeError("No bench_copies.csv: attach E5 step 1's output as input 3, or turn the internet on") from ex
with open(copies_path, encoding="utf-8") as f:
    bench_copies = {r["bench_file"]: r for r in csv.DictReader(f)}
COPIED = {"same pixels", "near copy"}        # E5 step 1's counted tiers; "review" matches are only pictured there


def copy_flags(r):
    """Each of the pair's two images whose best Instruct match (E5 step 1) is a copy, and where that image is."""
    flags = []
    for role, name in (("after", r["post"]), ("before", r["pre"])):
        c = bench_copies.get(name)
        if c and c["tier"] in COPIED:
            where = ("training" if c["instruct_file"] in TRAIN_FILES else
                     "validation" if c["instruct_file"] in VAL_FILES else "Instruct, outside the pool")
            flags.append({"role": role, "tier": c["tier"], "copy_of": c["instruct_file"], "where": where})
    return flags


for r in test:
    r["copies"] = copy_flags(r)
    r["copy_left_out"] = any(c["where"] in ("training", "validation") for c in r["copies"])
left_train = sum(any(c["where"] == "training" for c in r["copies"]) for r in test)
left_val = sum(r["copy_left_out"] and not any(c["where"] == "training" for c in r["copies"]) for r in test)
kept_outside = sum(bool(r["copies"]) and not r["copy_left_out"] for r in test)
COPY_RESULT = {"source": copies_source, "left_out": left_train + left_val, "left_out_training": left_train,
               "left_out_validation": left_val, "kept_copy_outside_pool": kept_outside,
               "by_kind": Counter("; ".join(sorted(f"{c['role']} {c['tier']} ({c['where']})" for c in r["copies"]))
                                  for r in test if r["copies"]),
               "events": Counter(event_and_source(r["post"])[0] for r in test if r["copy_left_out"])}
print(f"✓ Copies (from {copies_source}): {COPY_RESULT['left_out']} test pairs left out "
      f"({left_train} copy a training pair, {left_val} a validation pair); "
      f"{kept_outside} kept, copying an Instruct image outside the pool")
for kind, n in COPY_RESULT["by_kind"].most_common():
    print(f"  {n:4d}  {kind}")
print(f"  Events of the left-out pairs: {dict(COPY_RESULT['events'].most_common())}")

✓ Copies (from E5 step 1's Kaggle output (/kaggle/input/notebooks/abrar2222864642/e5-step1-hash-check/E5-step1/bench_copies.csv)): 57 test pairs left out (56 copy a training pair, 1 a validation pair); 2 kept, copying an Instruct image outside the pool
    31  after near copy (training); before same pixels (training)
    22  before same pixels (training)
     3  after same pixels (training)
     2  before same pixels (Instruct, outside the pool)
     1  after near copy (validation); before same pixels (validation)
  Events of the left-out pairs: {'woolsey_fire': 43, 'socal_fire': 11, 'pinery_bushfire': 3}


In [6]:
# ── Cell 6: Outlines drawn alike? (rule 5), then the results against the expectations, and the output files ──
import pandas as pd

for r in test:                         # a test pair is clean if it passed rules 1, 2 and 4
    s = TEST_STATS.get(r["post"])
    reasons = list(r["problems"])
    if s is None and not r["problems"]:
        reasons.append("not checked (smoke test)")
    elif s is not None and not passes(s):
        reasons.append("mask check failed")
    elif s is not None and not s["saved"]:
        reasons.append("merged mask not saved")
    if r["copy_left_out"]:
        reasons.append("copies a " + "/".join(sorted({c["where"] for c in r["copies"] if c["where"] != "Instruct, outside the pool"}))
                       + " image")
    r["left_out_because"] = "; ".join(reasons)
    r["clean"] = not reasons
CLEAN = [r for r in test if r["clean"]]


def event_table(rows, stats):
    pairs, areas, shares = Counter(), defaultdict(list), defaultdict(list)
    for r in rows:
        s = stats.get(r["post"])
        if s is None:
            continue
        ev = event_and_source(r["post"])[0]
        pairs[ev] += 1
        areas[ev] += s["piece_areas"]
        shares[ev].append(s["building_px"] / (1024 * 1024))
    return pairs, areas, shares


tp, ta, ts = event_table(CLEAN, TEST_STATS)
vp, va, vs = event_table(val_pool, VAL_STATS)
style_rows = []
for ev in sorted(set(tp) | set(vp)):
    row = {"event": ev, "val_pairs": vp[ev], "test_pairs": tp[ev],
           "val_median_piece_px": float(np.median(va[ev])) if va[ev] else None,
           "test_median_piece_px": float(np.median(ta[ev])) if ta[ev] else None,
           "val_median_building_share": round(float(np.median(vs[ev])), 4) if vs[ev] else None,
           "test_median_building_share": round(float(np.median(ts[ev])), 4) if ts[ev] else None}
    row["compared"] = vp[ev] >= STYLE_MIN_PAIRS and tp[ev] >= STYLE_MIN_PAIRS and bool(va[ev]) and bool(ta[ev])
    row["piece_ratio"] = round(row["test_median_piece_px"] / row["val_median_piece_px"], 3) if row["compared"] else None
    row["within_range"] = (STYLE_LOW <= row["piece_ratio"] <= STYLE_HIGH) if row["compared"] else None
    style_rows.append(row)
compared = [r for r in style_rows if r["compared"]]
style_share = sum(r["within_range"] for r in compared) / len(compared) if compared else None
STYLE_PASS = style_share is not None and style_share >= STYLE_SHARE

full = not SMOKE_TEST
checks = {
    "1. inventory: 2,067 pairs (1,030 / 558 / 479), no file missing":
        "yes" if (len(test), dict(by_masks), len(inventory_problems)) == (EXPECTED["test_pairs"], EXPECTED["test_by_masks"], 0)
        else f"NO ({len(test)} pairs, {dict(sorted(by_masks.items()))}, {len(inventory_problems)} with a missing file)",
    "2. every checked test pair's masks like the training masks":
        ("yes" if TEST_MASKS["fail_rule_2"] == 0 else f"NO ({TEST_MASKS['fail_rule_2']} pairs fail)") + ("" if full else " (smoke test: 40 pairs)"),
    "3. overlapping pixels at most 0.5% of building pixels":
        ("yes" if TEST_MASKS["overlap_share"] <= OVERLAP_LIMIT else f"NO ({TEST_MASKS['overlap_share']:.2%})")
        + f"; {TEST_MASKS['overlap_px']:,} pixels" + ("" if full else " (smoke test: 40 pairs)"),
    "4. copies: 57 left out (56 training, 1 validation)":
        "yes" if (COPY_RESULT["left_out"], left_train, left_val) == (EXPECTED["left_out_copies"], EXPECTED["left_out_training"],
                                                                     EXPECTED["left_out_validation"])
        else f"NO ({COPY_RESULT['left_out']}: {left_train} training, {left_val} validation)",
    "4. clean test pairs: 2,010":
        ("yes" if len(CLEAN) == EXPECTED["clean"] else f"NO ({len(CLEAN):,})") if full else "n/a (smoke test)",
    "5. outlines drawn alike (piece-area ratio 0.67-1.5 in at least 80% of shared events)":
        (f"{'yes' if STYLE_PASS else 'NO'} ({sum(r['within_range'] for r in compared)} of {len(compared)} events)"
         if compared else "n/a (no event with 20 pairs in both)") + ("" if full else " (smoke test)"),
}

# ── Output files ──
test_out = []
for r in test:
    s = TEST_STATS.get(r["post"], {})
    ev, src = event_and_source(r["post"])
    test_out.append({"after_image": r["post"], "before_image": r["pre"], "mask_file": r["post"], "mask_name_in_benchmark": r["mask_name"],
                     "event": ev, "source": src, "classes_asked": "+".join(map(str, r["referenced"])),
                     "clean": r["clean"], "left_out_because": r["left_out_because"],
                     "copies": "; ".join(f"{c['role']} {c['tier']} of {c['copy_of']} ({c['where']})" for c in r["copies"]),
                     "mask_checks_passed": passes(s) if s else None, "overlap_px": s.get("overlap_px"),
                     "building_px": s.get("building_px"),
                     **{f"{n}_px": (s.get("class_px") or [None] * 3)[k] for k, n in enumerate(("intact", "damaged", "destroyed"))},
                     **{f"{n}_pieces": (s.get("pieces") or [None] * 3)[k] for k, n in enumerate(("intact", "damaged", "destroyed"))}})
pd.DataFrame(test_out).to_csv(OUT_DIR / "test_pairs.csv", index=False)
pd.DataFrame([{k: t[k] for k in ("after_image", "before_image", "mask_file", "event", "source")}
              for t in test_out if t["clean"]]).to_csv(OUT_DIR / "test_clean_pairs.csv", index=False)
pd.DataFrame([{"after_image": p, "event": event_and_source(p)[0], "overlap_px": s["overlap_px"], "building_px": s["building_px"],
               "intact_px": s["class_px"][0], "damaged_px": s["class_px"][1], "destroyed_px": s["class_px"][2],
               "pieces": sum(s["pieces"]), "mask_checks_passed": passes(s)} for p, s in VAL_STATS.items()]
             ).to_csv(OUT_DIR / "val_pairs_stats.csv", index=False)
style_df = pd.DataFrame(style_rows).sort_values(["compared", "test_pairs"], ascending=False)
style_df.to_csv(OUT_DIR / "style_by_event.csv", index=False)

clean_events = Counter(event_and_source(r["post"])[0] for r in CLEAN)
clean_px = [sum(TEST_STATS[r["post"]]["class_px"][k] for r in CLEAN) for k in range(3)]
summary = {"task": "E6", "smoke_test": SMOKE_TEST, "test_pairs": len(test), "by_masks_asked": dict(by_masks),
           "sar_after_images_skipped": sar_images, "inventory_problems": len(inventory_problems),
           "test_masks": TEST_MASKS, "validation_masks": VAL_MASKS, "copies": {**COPY_RESULT, "by_kind": dict(COPY_RESULT["by_kind"]),
           "events": dict(COPY_RESULT["events"])}, "clean_pairs": len(CLEAN),
           "clean_class_px": {"intact": clean_px[0], "damaged": clean_px[1], "destroyed": clean_px[2]},
           "clean_by_event": dict(clean_events.most_common()), "style_events_compared": len(compared),
           "style_share_within_range": style_share, "style_pass": STYLE_PASS, "checks": checks}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, indent=1, default=int), encoding="utf-8")


def as_markdown(df):
    try:
        return df.to_markdown(index=False)
    except ImportError:              # `tabulate` missing: a plain-text table instead
        return "```\n" + df.to_string(index=False) + "\n```"


lines = [
    "# E6: the test split's ground truth", "",
    f"{'**SMOKE TEST** (all inventory and copy checks, masks of 40 test and 40 validation pairs): checks the code, not the answer. ' if SMOKE_TEST else ''}"
    f"The benchmark's optical building-damage segmentation entries give **{len(test):,} test pairs** "
    f"({sar_images:,} SAR after-images skipped: our models read colour photos).", "",
    "## Clean test pairs", "",
    f"- **{len(CLEAN):,} clean** of {len(test):,}. Left out: {len(inventory_problems)} with a missing file, "
    f"{TEST_MASKS['fail_rule_2']} failing the mask checks, {COPY_RESULT['left_out']} copying a segmentation-pool image "
    f"({left_train} training, {left_val} validation).",
    f"- Pixels in the clean pairs: intact {clean_px[0]:,}, damaged {clean_px[1]:,}, destroyed {clean_px[2]:,}.",
    f"- By event: {', '.join(f'{e} {n}' for e, n in clean_events.most_common())}.", "",
    "## The masks", "",
    as_markdown(pd.DataFrame([{"set": "test", **{k: v for k, v in TEST_MASKS.items() if k not in ('class_px', 'pieces')}},
                              {"set": "validation", **{k: v for k, v in VAL_MASKS.items() if k not in ('class_px', 'pieces')}}])), "",
    "Merged with the training rule (R11's Cell 4): intact, then damaged, then destroyed, a later class overwriting an earlier one. "
    "`test_masks_4class.zip` holds one PNG per checked test pair (0 background, 1 intact, 2 damaged, 3 destroyed), named after its "
    "after-image; use only the pairs in `test_clean_pairs.csv`.", "",
    "## Copies of segmentation-pool images", "",
    f"From {copies_source}: E5 step 1 compared every benchmark image with every Instruct image (same pixels; near copy: dHash within 6 "
    "of 64 bits and 32 x 32 thumbnails correlating at 0.95 or more) and kept each benchmark image's best match.", "",
    *[f"- {n}: {k}" for k, n in COPY_RESULT["by_kind"].most_common()], "",
    f"Events of the left-out pairs: {', '.join(f'{e} {n}' for e, n in COPY_RESULT['events'].most_common())}.", "",
    "## Outlines drawn alike? Median building-piece area by event", "",
    as_markdown(style_df[style_df["compared"]][["event", "val_pairs", "test_pairs", "val_median_piece_px", "test_median_piece_px",
                                                 "piece_ratio", "within_range", "val_median_building_share",
                                                 "test_median_building_share"]]), "",
    "## Against the rules and expectations (fixed before the run, 4 Nov)", "",
    *[f"- {k}: **{v}**" for k, v in checks.items()], "",
    "## Files", "",
    "- `test_pairs.csv`: every optical test pair, its checks, copies, pixels and pieces, and whether it's clean",
    "- `test_clean_pairs.csv`: the clean pairs (after-image, before-image, mask file, event, source)",
    "- `val_pairs_stats.csv`, `style_by_event.csv`, `summary.json`",
    "- `test_masks_4class.zip`: the merged four-class masks",
]
(OUT_DIR / "README.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
shutil.make_archive(str(OUT_DIR / "test_masks_4class"), "zip", MASK_OUT)
n_zipped = len(list(MASK_OUT.iterdir()))
shutil.rmtree(MASK_OUT)                # the zip holds them; one copy is enough to download
print(f"✓ {len(CLEAN):,} clean test pairs; {n_zipped:,} merged masks in test_masks_4class.zip")
for k, v in checks.items():
    print(f"  {k}: {v}")
print(f"\nOutputs in {OUT_DIR}: " + ", ".join(sorted(p.name for p in OUT_DIR.iterdir())))

✓ 40 clean test pairs; 40 merged masks in test_masks_4class.zip
  1. inventory: 2,067 pairs (1,030 / 558 / 479), no file missing: yes
  2. every checked test pair's masks like the training masks: yes (smoke test: 40 pairs)
  3. overlapping pixels at most 0.5% of building pixels: yes; 0 pixels (smoke test: 40 pairs)
  4. copies: 57 left out (56 training, 1 validation): yes
  4. clean test pairs: 2,010: n/a (smoke test)
  5. outlines drawn alike (piece-area ratio 0.67-1.5 in at least 80% of shared events): n/a (no event with 20 pairs in both) (smoke test)

Outputs in /kaggle/working/E6: README.md, style_by_event.csv, summary.json, test_clean_pairs.csv, test_masks_4class.zip, test_pairs.csv, val_pairs_stats.csv
